# NumPy Tutorial — Thesis-Focused
**Preparation Phase · Bachelor's Thesis: Benchmarking Tabular Data Augmentation Techniques for Deep Clustering**

Background: basic Python ✓ · numpy ← you are here · pandas · matplotlib · scikit-learn

NumPy is the foundation of Python's scientific stack.
Pandas, scikit-learn, PyTorch — all of them use numpy arrays internally.
Learning numpy first means every other library will feel familiar.

| § | Topic | Where you will use it |
|---|-------|-----------------------|
| 1 | Creating and inspecting arrays | Setting up data structures |
| 2 | Indexing: rows, columns, slices | Accessing samples and features |
| 3 | Boolean indexing (masking) | Filtering by cluster, feature masking augmentation |
| 4 | Array math and aggregation | Standardisation, loss computation |
| 5 | Combining and reshaping | Building augmented pairs for contrastive training |
| 6 | Random number generation | Noise augmentation, reproducibility |
| ★ | **Capstone** | Implement SCARF-style feature masking augmentation |

**Structure of each section:** explanation → worked example → mini-example you can run → your exercise → solution.

---
## Before You Run Anything: Understanding the Shared Data

### What is a numpy array — and why not a Python list?

A Python list can hold anything: `[1, 'hello', True, 3.14]` — mixed types, flexible.
A numpy array holds only **one type** (e.g. all floats, or all integers).
This restriction is what makes numpy fast: it can process the entire array in one
compiled operation instead of looping through elements in Python.

Concretely: adding 1 to every element of a 10 000-element array takes
microseconds with numpy and milliseconds with a Python loop — 100× faster.
Your thesis datasets have thousands of rows and you will do this constantly.

### The universal convention in machine learning

A 2D numpy array representing a dataset is always arranged as:

```
rows    = samples  (one row per data point)
columns = features (one column per measured variable)
```

So `X.shape = (120, 6)` means 120 data points, each described by 6 features.
scikit-learn, PyTorch, and every other library assumes this layout.

### Connection to pandas

A pandas DataFrame is essentially a numpy array with named columns and a row index.
You convert between them with:
```python
X_np = df.to_numpy()   # DataFrame → numpy array
```
Your thesis pipeline does exactly this after preprocessing:
preprocess the DataFrame in pandas, then hand the numpy array to scikit-learn / PyTorch.

### Variable guide

| Variable | Shape | What it represents |
|----------|-------|-------------------|
| `X_small` | (6, 4) | A tiny tabular dataset — small enough to inspect by eye |
| `feature_names` | list | Column names for `X_small`: Age, Income, HasDegree, YearsExp |
| `labels_small` | (6,) | Cluster label (0 or 1) for each row of `X_small` |
| `X` | (120, 6) | A larger dataset for augmentation exercises — already standardised |
| `labels` | (120,) | Cluster label (0, 1, or 2) for each row of `X` |

In [1]:
import numpy as np

# Fixed-seed random generator — explained fully in §6
rng = np.random.default_rng(42)


# ── SMALL DATASET (§1 – §4) ────────────────────────────────────────────────
# 6 samples × 4 features. Small enough to read by eye.
# Columns: [Age, Income (k€), HasDegree (0/1), YearsExp]
X_small = np.array([
    [23.0,  42.0, 0.0,  1.0],   # sample 0
    [45.0,  83.0, 1.0,  8.0],   # sample 1
    [31.0,  56.0, 1.0,  3.0],   # sample 2
    [52.0,  97.0, 1.0, 12.0],   # sample 3
    [28.0,  38.0, 0.0,  2.0],   # sample 4
    [39.0,  61.0, 0.0,  6.0],   # sample 5
])
# X_small.shape = (6, 4): 6 rows (samples) × 4 columns (features)

feature_names = ['Age', 'Income', 'HasDegree', 'YearsExp']

# One label per sample: cluster 0 or cluster 1
labels_small = np.array([0, 1, 0, 1, 0, 1])


# ── LARGER DATASET (§5 – §6 + Capstone) ───────────────────────────────────
# 120 samples × 6 features, already standardised (mean ≈ 0, std ≈ 1 per column).
# Simulates what your tabular data looks like after StandardScaler.
X = rng.standard_normal((120, 6))
# X.shape = (120, 6)

# np.repeat([0, 1, 2], 40) → [0, 0, ...(40×), 1, 1, ...(40×), 2, 2, ...(40×)]
# One cluster label per sample: 3 clusters × 40 samples = 120 labels
labels = np.repeat([0, 1, 2], 40)


print('X_small shape:', X_small.shape, '  dtype:', X_small.dtype)
print('labels_small: ', labels_small)
print('X shape:      ', X.shape,       '  dtype:', X.dtype)
print('labels shape: ', labels.shape)

X_small shape: (6, 4)   dtype: float64
labels_small:  [0 1 0 1 0 1]
X shape:       (120, 6)   dtype: float64
labels shape:  (120,)


---
## §1 Creating and Inspecting Arrays

### The core attributes

Every numpy array has three attributes you will check constantly:

| Attribute | What it tells you | Example |
|-----------|------------------|---------|
| `.shape` | Dimensions as a tuple — `(rows, cols)` for 2D | `(6, 4)` |
| `.dtype` | The data type of every element | `float64` |
| `.size` | Total number of elements (rows × cols) | `24` |

A quick rule: if `.shape` surprises you, something went wrong upstream.
Checking `.shape` after every major operation is good practice.

### Creation functions

| Function | What it creates |
|----------|-----------------|
| `np.array([...])` | Array from existing Python data |
| `np.zeros((r, c))` | Array of all zeros, shape `(r, c)` |
| `np.ones((r, c))` | Array of all ones, shape `(r, c)` |
| `np.full((r, c), v)` | Array filled with value `v` |
| `np.arange(start, stop, step)` | Like `range()`, but returns an array |
| `np.linspace(start, stop, n)` | `n` evenly spaced values from start to stop (inclusive) |
| `np.eye(n)` | `n×n` identity matrix (1s on diagonal, 0s elsewhere) |

**Note on shape syntax:** numpy always wants shapes as a **tuple** — `(3, 4)`, not `3, 4`.
For a 1D array, the shape tuple has one element: `(10,)` (note the trailing comma).

In [ ]:
# ── §1 Worked example ────────────────────────────────────────────────────────

# Inspect the small dataset
print('=== X_small ===')
print(X_small)              # print the full array
print('shape:', X_small.shape)   # (6, 4): 6 rows, 4 columns
print('dtype:', X_small.dtype)   # float64: 64-bit floating point
print('size: ', X_small.size)    # 24: 6 × 4 total elements
print()

# Create arrays from scratch
zeros_3x4 = np.zeros((3, 4))    # 3 rows, 4 columns, all 0.0
print('zeros (3×4):')
print(zeros_3x4)
print()

# np.arange(start, stop, step): like Python's range() but returns an array
# Note: stop is excluded, just like range()
evens = np.arange(0, 10, 2)    # [0, 2, 4, 6, 8]
print('np.arange(0, 10, 2):', evens)
print()

# np.linspace(start, stop, n): n values equally spaced from start TO stop (stop IS included)
unit_interval = np.linspace(0.0, 1.0, 6)   # [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
print('np.linspace(0, 1, 6):', unit_interval)
print()

# np.eye(n): n×n identity matrix — diagonal is 1, everything else 0
# You saw this in the matplotlib notebook: 0.4 * np.eye(2)
identity_3 = np.eye(3)
print('np.eye(3):')
print(identity_3)

### Exercise 1

Create the following four arrays and print the shape of each:

1. A 4×3 matrix of all **zeros**
2. A 1D array of **5 evenly spaced values from 0.0 to 1.0** (use `np.linspace`)
3. A 3×3 array where **every element is 7.0** (use `np.full`)
4. A 4×4 **identity matrix**

The mini-example below shows the pattern for one of them — run it, then fill in the template.

In [ ]:
# ── §1 Mini-example: np.full ─────────────────────────────────────────────────
# np.full((rows, cols), fill_value) creates an array of the given shape
# where every element equals fill_value.

filled = np.full((2, 5), 3.14)   # 2 rows, 5 columns, all 3.14
print('np.full((2, 5), 3.14):')
print(filled)
print('shape:', filled.shape)    # (2, 5)
# Pattern: np.full((rows, cols), value) — same as this for your exercise.

In [2]:
# ── §1 Exercise 1 — template (replace ... with your code) ────────────────────

# 1. 4×3 matrix of zeros
a = np.zeros((4,3))   # np.zeros(...)
print('1. shape:', a.shape)

# 2. Five evenly spaced values from 0.0 to 1.0
b = np.linspace(0.0, 1.0, 5)  # np.linspace(...)
print('2. values:', b, '  shape:', b.shape)

# 3. 3×3 array where every element is 7.0
c = np.full((3,3), 7.0)   # np.full(...)
print('3.\n', c, '  shape:', c.shape)

# 4. 4×4 identity matrix
d = np.eye(4)   # np.eye(...)
print('4.\n', d, '  shape:', d.shape)

1. shape: (4, 3)
2. values: [0.   0.25 0.5  0.75 1.  ]   shape: (5,)
3.
 [[7. 7. 7.]
 [7. 7. 7.]
 [7. 7. 7.]]   shape: (3, 3)
4.
 [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]   shape: (4, 4)


In [3]:
# ── §1 SOLUTION ──────────────────────────────────────────────────────────────

a = np.zeros((4, 3))
print('1. shape:', a.shape)

b = np.linspace(0.0, 1.0, 5)
print('2. values:', b, '  shape:', b.shape)

c = np.full((3, 3), 7.0)
print('3.\n', c, '  shape:', c.shape)

d = np.eye(4)
print('4.\n', d, '  shape:', d.shape)

1. shape: (4, 3)
2. values: [0.   0.25 0.5  0.75 1.  ]   shape: (5,)
3.
 [[7. 7. 7.]
 [7. 7. 7.]
 [7. 7. 7.]]   shape: (3, 3)
4.
 [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]   shape: (4, 4)


---
## §2 Indexing: Rows, Columns, Slices

### The key rule: rows first, then columns

For a 2D array `A`, every index expression follows the pattern `A[row, col]`.
Think of it like a spreadsheet: you name the row first, then the column.

```
X_small has 6 rows (samples 0–5) and 4 columns (features 0–3)

           col 0   col 1    col 2    col 3
           Age     Income   HasDeg   YrsExp
row 0  →  [23.0,   42.0,    0.0,     1.0 ]
row 1  →  [45.0,   83.0,    1.0,     8.0 ]
row 2  →  [31.0,   56.0,    1.0,     3.0 ]
row 3  →  [52.0,   97.0,    1.0,    12.0 ]
row 4  →  [28.0,   38.0,    0.0,     2.0 ]
row 5  →  [39.0,   61.0,    0.0,     6.0 ]
```

| Expression | What you get |
|------------|--------------|
| `A[2, 1]` | Single element: row 2, col 1 → `56.0` |
| `A[2, :]` | All columns of row 2 → one sample: `[31, 56, 1, 3]` |
| `A[:, 1]` | All rows of col 1 → one feature: `[42, 83, 56, 97, 38, 61]` |
| `A[1:4, :]` | Rows 1, 2, 3 (stop is excluded) → 3 samples |
| `A[-1, :]` | Last row → sample 5 |
| `A[:, 0:2]` | Columns 0 and 1 → first 2 features |

**The colon `:` means "everything"** — `A[:, 1]` means "every row, column 1".
You can often leave it out: `A[2]` is the same as `A[2, :]` for 2D arrays.

In [ ]:
# ── §2 Worked example ────────────────────────────────────────────────────────

print('Full X_small:')
print(X_small)
print()

# Single element — row 2, column 1 (Income of sample 2)
elem = X_small[2, 1]
print('X_small[2, 1] (Income of sample 2):', elem)   # 56.0

# One full row — all features of sample 1
sample_1 = X_small[1, :]    # same as X_small[1]
print('X_small[1, :] (all features of sample 1):', sample_1)

# One full column — the Age feature for all samples
ages = X_small[:, 0]        # column 0 = Age
print('X_small[:, 0] (Age column):', ages)

# A slice of rows — samples 2, 3, 4 (rows 2 to 4, stop=5 is excluded)
subset = X_small[2:5, :]
print('X_small[2:5, :] (samples 2–4):')
print(subset)

# Negative index — the last sample
last = X_small[-1, :]
print('X_small[-1, :] (last sample):', last)

### Exercise 2

Using only indexing on `X_small` and `feature_names`, extract:

1. The **Income** of sample 3 (a single number)
2. The **YearsExp column** for all samples (a 1D array of 6 values)
3. The **first 3 samples**, all features (a 2D sub-array)
4. The **last 2 samples**, only columns 0 and 1 (Age and Income)

Run the mini-example first — it shows how to think about row/column indexing.

In [ ]:
# ── §2 Mini-example: thinking out loud ──────────────────────────────────────
# 'I want the HasDegree column (column 2) for samples 1 and 2 only (rows 1 and 2).'
# → rows: 1 and 2 → slice 1:3  (stop is excluded, so 1:3 gives rows 1, 2)
# → cols: column 2 only → index 2

result = X_small[1:3, 2]   # rows 1–2, column 2
print('HasDegree for samples 1 and 2:', result)   # [1.0, 1.0]

# Reading this back: 'rows 1 to 2 (exclusive of 3), column 2'
# Apply the same thinking to Exercise 2 below.

In [ ]:
# ── §2 Exercise 2 — template ─────────────────────────────────────────────────

# 1. Income of sample 3 — one number
#    Income is column 1 (see feature_names)
income_s3 = X_small[...]   # ← row 3, col 1
print('1. Income of sample 3:', income_s3)

# 2. YearsExp column — all rows, column 3
years_exp = X_small[...]   # ← all rows, col 3
print('2. YearsExp column:', years_exp)

# 3. First 3 samples — rows 0, 1, 2; all columns
first_3 = X_small[...]     # ← slice rows 0:3, all cols
print('3. First 3 samples:')
print(first_3)

# 4. Last 2 samples — rows 4, 5; columns 0 and 1 only
last2_age_income = X_small[...]   # ← slice rows -2:, cols 0:2
print('4. Last 2 samples, Age + Income:')
print(last2_age_income)

In [ ]:
# ── §2 SOLUTION ──────────────────────────────────────────────────────────────

income_s3 = X_small[3, 1]
print('1. Income of sample 3:', income_s3)   # 97.0

years_exp = X_small[:, 3]
print('2. YearsExp column:', years_exp)

first_3 = X_small[0:3, :]
print('3. First 3 samples:')
print(first_3)

last2_age_income = X_small[-2:, 0:2]
print('4. Last 2 samples, Age + Income:')
print(last2_age_income)

---
## §3 Boolean Indexing — Masking

This is the most important section for your thesis.
The mask pattern appears in almost every step of your pipeline:
filtering samples by cluster label, zeroing out features during augmentation,
and selecting results above a threshold.

### How it works — three steps

**Step 1: Create a boolean array** by applying a comparison to an array.
```python
ages = X_small[:, 0]          # [23, 45, 31, 52, 28, 39]
mask = ages > 30              # [False, True, True, True, False, True]
```
The result has the same shape as `ages`, but every element is `True` or `False`.

**Step 2: Use the boolean array to select rows (or elements).**
```python
X_small[mask]    # keeps only the rows where mask is True → rows 1, 2, 3, 5
```

**Step 3: Optionally, assign new values to the selected positions.**
```python
X_copy[mask, 0] = 0.0   # set Age to 0 for samples where Age > 30
```

### Combining conditions

Use `&` (AND) and `|` (OR) — **not** Python's `and` / `or` (those don't work on arrays).
Always wrap each condition in parentheses:
```python
mask = (ages > 30) & (X_small[:, 1] > 50)   # Age > 30 AND Income > 50
```

### Thesis connections

| Pattern | Where it appears |
|---------|------------------|
| `X[labels == k]` | Select all samples in cluster k |
| `rng.random(n_features) < 0.3` | Random feature mask: True with 30% probability |
| `X_aug[i, feat_mask] = 0.0` | Zero out masked features (SCARF augmentation) |

In [4]:
# ── §3 Worked example ────────────────────────────────────────────────────────

print('X_small (Age is column 0):')
print(X_small)
print()

# Step 1: create a boolean array
age_mask = X_small[:, 0] > 30   # True where Age > 30
print('age_mask (Age > 30):        ', age_mask)   # [F, T, T, T, F, T]

# Step 2: filter rows
old_samples = X_small[age_mask]   # only rows where mask is True
print('Samples with Age > 30 (rows 1,2,3,5):')
print(old_samples)
print()

# Compound condition: Age > 30 AND Income > 60
compound_mask = (X_small[:, 0] > 30) & (X_small[:, 1] > 60)
print('compound_mask (Age > 30 AND Income > 60):', compound_mask)
print('Matching samples:')
print(X_small[compound_mask])
print()

# Thesis pattern: select all samples in cluster 0
cluster0 = X_small[labels_small == 0]
print('Samples in cluster 0 (labels_small == 0):')
print(cluster0)

X_small (Age is column 0):
[[23. 42.  0.  1.]
 [45. 83.  1.  8.]
 [31. 56.  1.  3.]
 [52. 97.  1. 12.]
 [28. 38.  0.  2.]
 [39. 61.  0.  6.]]

age_mask (Age > 30):         [False  True  True  True False  True]
Samples with Age > 30 (rows 1,2,3,5):
[[45. 83.  1.  8.]
 [31. 56.  1.  3.]
 [52. 97.  1. 12.]
 [39. 61.  0.  6.]]

compound_mask (Age > 30 AND Income > 60): [False  True False  True False  True]
Matching samples:
[[45. 83.  1.  8.]
 [52. 97.  1. 12.]
 [39. 61.  0.  6.]]

Samples in cluster 0 (labels_small == 0):
[[23. 42.  0.  1.]
 [31. 56.  1.  3.]
 [28. 38.  0.  2.]]


### Exercise 3

Two tasks:

1. From `X_small`, select all samples where **Income > 50 AND HasDegree == 1**.
   How many samples match?

2. **Feature masking (simplified SCARF):** make a copy of `X_small`, then
   set the **Age column (col 0) to 0.0** for all samples in **cluster 1** (`labels_small == 1`).
   Print the original and the modified copy side by side.

The mini-example shows how to modify selected positions without touching the original.

In [5]:
# ── §3 Mini-example: modify with a mask (step 3) ────────────────────────────

# IMPORTANT: always .copy() before modifying — explained fully in §5
X_copy = X_small.copy()   # independent copy — changes here won't affect X_small

# Set YearsExp (col 3) to 0 for the last 2 samples (mask = age < 30)
young_mask = X_copy[:, 0] < 30   # True for samples 0 and 4 (Age 23 and 28)
X_copy[young_mask, 3] = 0.0      # set YearsExp to 0 for those samples

print('Original X_small[:, 3] (YearsExp):', X_small[:, 3])
print('Modified X_copy[:, 3]  (YearsExp):', X_copy[:, 3])
# Notice: X_small is unchanged — the .copy() protected it.

Original X_small[:, 3] (YearsExp): [ 1.  8.  3. 12.  2.  6.]
Modified X_copy[:, 3]  (YearsExp): [ 0.  8.  3. 12.  0.  6.]


In [6]:
# ── §3 Exercise 3 — template ─────────────────────────────────────────────────

# Task 1: Income > 50 AND HasDegree == 1
mask_task1 = (X_small[:, 1] > 50) & (X_small[:, 2 ] == 1)  # ← (X_small[:, 1] > 50) & (...)
selected = X_small[mask_task1]
print('Task 1 — matching samples:')
print(selected)
print('Count:', len(selected))
print()

# Task 2: zero out Age (col 0) for cluster 1 samples
X_masked = X_small.copy()         # start from a copy!
cluster1_mask = labels_small == 1            # ← labels_small == 1
X_masked[cluster1_mask, 0] = 0.0  # ← this line is correct; just fill in the mask above

print('Task 2 — Age column before masking:', X_small[:, 0])
print('Task 2 — Age column after  masking:', X_masked[:, 0])

Task 1 — matching samples:
[[45. 83.  1.  8.]
 [31. 56.  1.  3.]
 [52. 97.  1. 12.]]
Count: 3

Task 2 — Age column before masking: [23. 45. 31. 52. 28. 39.]
Task 2 — Age column after  masking: [23.  0. 31.  0. 28.  0.]


In [7]:
# ── §3 SOLUTION ──────────────────────────────────────────────────────────────

# Task 1
mask_task1 = (X_small[:, 1] > 50) & (X_small[:, 2] == 1)
selected = X_small[mask_task1]
print('Task 1 — matching samples (Income > 50 AND HasDegree == 1):')
print(selected)
print('Count:', len(selected))   # 3 samples
print()

# Task 2
X_masked = X_small.copy()
cluster1_mask = labels_small == 1
X_masked[cluster1_mask, 0] = 0.0

print('Task 2 — Age column before masking:', X_small[:, 0])
print('Task 2 — Age column after  masking:', X_masked[:, 0])
# Samples 1, 3, 5 (cluster 1) now have Age = 0.0

Task 1 — matching samples (Income > 50 AND HasDegree == 1):
[[45. 83.  1.  8.]
 [31. 56.  1.  3.]
 [52. 97.  1. 12.]]
Count: 3

Task 2 — Age column before masking: [23. 45. 31. 52. 28. 39.]
Task 2 — Age column after  masking: [23.  0. 31.  0. 28.  0.]


---
## §4 Array Math and Aggregation

### Vectorised operations — no loops needed

When you write `array + 1`, numpy adds 1 to **every element at once** — no Python loop.
This is called a **vectorised operation** and it is 10–100× faster than a loop.

| Expression | What happens |
|------------|--------------|
| `A + B` | Add corresponding elements (A and B must have the same shape) |
| `A * 2` | Multiply every element by 2 |
| `A ** 2` | Square every element |
| `np.sqrt(A)` | Square root of every element |
| `np.exp(A)` | e^x for every element |
| `np.abs(A)` | Absolute value of every element |

### Aggregation — summarising an array

| Method | What it computes |
|--------|-----------------|
| `A.mean()` | Average of all elements |
| `A.std()` | Standard deviation of all elements |
| `A.sum()` | Sum of all elements |
| `A.min()` / `A.max()` | Smallest / largest element |
| `np.argmin(A)` / `np.argmax(A)` | **Index** of the smallest / largest element |

### The `axis` parameter — the most non-obvious concept

By default, `.mean()` averages over **all** elements and returns one number.
But in machine learning you almost always want **one value per feature** (column mean)
or **one value per sample** (row mean). The `axis` argument controls this:

```
           col 0  col 1  col 2
row 0  →  [  2,     4,     6  ]
row 1  →  [  8,    10,    12  ]

.mean(axis=0) → [5, 7, 9]      collapse rows → one value per column (per feature)
.mean(axis=1) → [4, 10]        collapse cols → one value per row (per sample)
.mean()       → 7.0            collapse everything → one grand average
```

### Thesis connection: StandardScaler from scratch

scikit-learn's `StandardScaler` does exactly this:
```python
mean = X.mean(axis=0)        # one mean per feature
std  = X.std(axis=0)         # one std  per feature
X_scaled = (X - mean) / std  # subtract and divide — element-wise broadcasting
```
After scaling, each feature has mean ≈ 0 and std ≈ 1.
Understanding this helps you debug preprocessing issues.

In [8]:
# ── §4 Worked example ────────────────────────────────────────────────────────

# Element-wise operations
ages = X_small[:, 0]    # [23, 45, 31, 52, 28, 39]
print('Ages:          ', ages)
print('Ages + 1:      ', ages + 1)       # adds 1 to every element
print('Ages / 100:    ', ages / 100)     # divides every element by 100
print('Ages ** 2:     ', ages ** 2)      # squares every element
print()

# Aggregation on a 1D array
print('Mean age:      ', ages.mean())    # average
print('Std of ages:   ', ages.std().round(2))
print('Min / max age: ', ages.min(), '/', ages.max())
print('Index of max:  ', np.argmax(ages))  # → 3 (sample 3 has Age=52)
print()

# axis=0 on the full 2D dataset
col_means = X_small.mean(axis=0)   # one mean per column (per feature)
col_stds  = X_small.std(axis=0)    # one std  per column
print('Column means (one per feature):   ', col_means.round(2))
print('Column stds  (one per feature):   ', col_stds.round(2))
print()

# StandardScaler-style normalisation of X_small
X_scaled = (X_small - col_means) / col_stds
print('X_scaled (first 3 rows):')
print(X_scaled[:3].round(3))
print('Verify — scaled column means (should be ≈ 0):', X_scaled.mean(axis=0).round(10))
print('Verify — scaled column stds  (should be ≈ 1):', X_scaled.std(axis=0).round(3))

Ages:           [23. 45. 31. 52. 28. 39.]
Ages + 1:       [24. 46. 32. 53. 29. 40.]
Ages / 100:     [0.23 0.45 0.31 0.52 0.28 0.39]
Ages ** 2:      [ 529. 2025.  961. 2704.  784. 1521.]

Mean age:       36.333333333333336
Std of ages:    10.03
Min / max age:  23.0 / 52.0
Index of max:   3

Column means (one per feature):    [36.33 62.83  0.5   5.33]
Column stds  (one per feature):    [10.03 21.11  0.5   3.82]

X_scaled (first 3 rows):
[[-1.33  -0.987 -1.    -1.136]
 [ 0.864  0.955  1.     0.699]
 [-0.532 -0.324  1.    -0.612]]
Verify — scaled column means (should be ≈ 0): [-0. -0.  0.  0.]
Verify — scaled column stds  (should be ≈ 1): [1. 1. 1. 1.]


### Exercise 4

The `axis` parameter is the trickiest part of this section.
The mini-example below shows what `axis=0` vs `axis=1` produce on a tiny array.
Run it, make sure it makes sense, then complete the template.

**Your task:**
1. Compute the **per-feature mean and std** of `X_small` (columns 0 and 1 only — Age and Income).
2. Standardise only those two columns manually.
3. Verify the result has mean ≈ 0 and std ≈ 1 per column.

In [ ]:
# ── §4 Mini-example: axis=0 vs axis=1 ───────────────────────────────────────

tiny = np.array([[2.0, 4.0, 6.0],
                 [8.0, 10.0, 12.0]])
# Shape: (2 rows, 3 cols)

print('tiny:')
print(tiny)
print()

# axis=0: collapse ROWS → compute one value per COLUMN
# Average of [2, 8]=5, [4, 10]=7, [6, 12]=9
print('tiny.mean(axis=0) — one mean per COLUMN:', tiny.mean(axis=0))   # [5, 7, 9]

# axis=1: collapse COLUMNS → compute one value per ROW
# Average of [2, 4, 6]=4.0, [8, 10, 12]=10.0
print('tiny.mean(axis=1) — one mean per ROW:   ', tiny.mean(axis=1))   # [4, 10]

# In ML: axis=0 is almost always what you want (per-feature statistics)

In [ ]:
# ── §4 Exercise 4 — template ─────────────────────────────────────────────────

# Extract Age and Income columns
X_num = X_small[:, 0:2]   # shape (6, 2)

# 1. Per-feature mean and std (one value per column)
mean_num = ...   # ← .mean(axis=0)
std_num  = ...   # ← .std(axis=0)
print('Column means:', mean_num.round(2))
print('Column stds: ', std_num.round(2))

# 2. Standardise
X_num_scaled = ...   # ← (X_num - mean_num) / std_num
print('Scaled (first 3 rows):')
print(X_num_scaled[:3].round(3))

# 3. Verify
print('Scaled means (should be ≈ 0):', X_num_scaled.mean(axis=0).round(10))
print('Scaled stds  (should be ≈ 1):', X_num_scaled.std(axis=0).round(3))

In [ ]:
# ── §4 SOLUTION ──────────────────────────────────────────────────────────────

X_num = X_small[:, 0:2]

mean_num = X_num.mean(axis=0)
std_num  = X_num.std(axis=0)
print('Column means:', mean_num.round(2))
print('Column stds: ', std_num.round(2))

X_num_scaled = (X_num - mean_num) / std_num
print('Scaled (first 3 rows):')
print(X_num_scaled[:3].round(3))

print('Scaled means (should be ≈ 0):', X_num_scaled.mean(axis=0).round(10))
print('Scaled stds  (should be ≈ 1):', X_num_scaled.std(axis=0).round(3))

---
## §5 Combining and Reshaping

### Stacking arrays

| Function | What it does | Requires |
|----------|-------------|----------|
| `np.vstack([A, B])` | Stack A **above** B — adds rows | Same number of **columns** |
| `np.hstack([A, B])` | Stack A **left of** B — adds columns | Same number of **rows** |
| `np.concatenate([A, B], axis=0)` | General version of vstack | Same columns |

Think of `vstack` as "paste a second spreadsheet below the first".
Think of `hstack` as "paste a second spreadsheet to the right of the first".

### Contrastive learning needs vstack

Your training loop will create **(original, augmented) pairs**:
```python
X_combined = np.vstack([X, X_aug])      # (240, 6): 120 originals + 120 augmented
y_combined = np.concatenate([labels, labels])  # (240,): same labels for both
```
The model learns that original[i] and augmented[i] should have similar embeddings.

### Why `.copy()` matters

This is a subtle but important point:

```python
X_aug = X          # ← WRONG: X_aug and X point to the same data in memory
X_aug[0, 0] = 99   # ← this also changes X[0, 0]!

X_aug = X.copy()   # ← CORRECT: X_aug is an independent copy
X_aug[0, 0] = 99   # ← X is unaffected
```

**Always `.copy()` before modifying**, especially inside augmentation functions.
Otherwise you corrupt your original data.

### Reshaping

`.reshape(new_shape)` reorganises elements into a different shape.
The total number of elements must stay the same.

```python
a = np.arange(12)       # [0, 1, 2, ..., 11]  shape: (12,)
a.reshape(3, 4)          # 3 rows × 4 cols      shape: (3, 4)
a.reshape(2, -1)         # 2 rows, auto cols    shape: (2, 6)  (-1 = 'figure it out')
a.reshape(-1, 1)         # column vector        shape: (12, 1)
```

In [ ]:
# ── §5 Worked example ────────────────────────────────────────────────────────

# Demonstrate .copy() — the danger without it
X_alias  = X_small        # NOT a copy — just another name for the same array
X_safe   = X_small.copy() # an independent copy

X_alias[0, 0] = 9999.0    # modifies the underlying data
print('X_small[0, 0] after modifying X_alias:', X_small[0, 0])  # also 9999 — oops!

X_safe[0, 0] = 8888.0     # modifies only X_safe
print('X_small[0, 0] after modifying X_safe: ', X_small[0, 0])  # still 9999 — X_small unchanged

# Restore X_small for the rest of the notebook
X_small[0, 0] = 23.0
print('X_small restored:', X_small[0, 0])
print()

# vstack: combine two datasets into one
A = np.array([[1.0, 2.0], [3.0, 4.0]])
B = np.array([[5.0, 6.0], [7.0, 8.0]])

stacked = np.vstack([A, B])   # A on top, B below
print('A:', A.shape, '  B:', B.shape, '  vstack:', stacked.shape)
print(stacked)
print()

# Thesis pattern: original + augmented → combined
# (Using X as the original — imagine X_aug is a modified copy)
X_aug_example = X + 0.1   # trivial augmentation: just add 0.1 to everything
X_combined = np.vstack([X, X_aug_example])     # (240, 6)
y_combined = np.concatenate([labels, labels])   # (240,)
print('X.shape:         ', X.shape)
print('X_aug.shape:     ', X_aug_example.shape)
print('X_combined.shape:', X_combined.shape)    # (240, 6): 120 + 120 rows
print('y_combined.shape:', y_combined.shape)    # (240,)

### Exercise 5

Run the mini-example to see vstack and hstack on tiny arrays.
Then complete the template:

1. Create a simple augmented version of `X_small`: multiply every element by 0.9 (a trivial "scaling augmentation").
2. Stack the original and augmented into one combined array using `np.vstack`.
3. Create a combined label array with `np.concatenate`.
4. Verify that the combined shape is `(12, 4)` and labels shape is `(12,)`.

In [ ]:
# ── §5 Mini-example: vstack vs hstack ───────────────────────────────────────

top    = np.array([[1, 2, 3]])
bottom = np.array([[4, 5, 6]])
left   = np.array([[10], [20]])
right  = np.array([[30], [40]])

# vstack: add rows (must have same number of columns)
print('vstack (adds rows):')
print(np.vstack([top, bottom]))   # shape (2, 3)

# hstack: add columns (must have same number of rows)
print('hstack (adds columns):')
print(np.hstack([left, right]))   # shape (2, 2)

In [ ]:
# ── §5 Exercise 5 — template ─────────────────────────────────────────────────

# 1. Trivial augmentation: scale every element by 0.9
#    Use .copy() first so X_small is not modified!
X_small_aug = X_small.copy() * ...   # ← multiply by 0.9

# 2. Stack original and augmented
X_small_combined = np.vstack([..., ...])   # ← X_small, then X_small_aug

# 3. Create combined labels
labels_combined = np.concatenate([..., ...])   # ← labels_small twice

# 4. Verify shapes
print('X_small shape:          ', X_small.shape)
print('X_small_aug shape:      ', X_small_aug.shape)
print('X_small_combined shape: ', X_small_combined.shape)  # should be (12, 4)
print('labels_combined shape:  ', labels_combined.shape)   # should be (12,)
print('labels_combined:        ', labels_combined)

In [ ]:
# ── §5 SOLUTION ──────────────────────────────────────────────────────────────

X_small_aug = X_small.copy() * 0.9
X_small_combined = np.vstack([X_small, X_small_aug])
labels_combined  = np.concatenate([labels_small, labels_small])

print('X_small shape:          ', X_small.shape)
print('X_small_aug shape:      ', X_small_aug.shape)
print('X_small_combined shape: ', X_small_combined.shape)   # (12, 4)
print('labels_combined shape:  ', labels_combined.shape)    # (12,)
print('labels_combined:        ', labels_combined)

---
## §6 Random Number Generation

### The modern interface: `np.random.default_rng(seed)`

Always use this — not the older `np.random.rand()` or `np.random.seed()`.
The modern interface is more predictable and easier to reason about.

```python
rng = np.random.default_rng(42)   # create a generator, locked to seed 42
```

### Why seeding matters for your thesis

Your benchmark will run each experiment with **5 different seeds**.
Without a fixed seed, two runs of the same code produce different results — impossible to reproduce.
With a fixed seed, the same seed always gives the same random numbers:

```python
rng_a = np.random.default_rng(42)
rng_b = np.random.default_rng(42)
# rng_a and rng_b will produce identical sequences.
```

### Key methods

| Method | What it generates |
|--------|-----------------|
| `rng.standard_normal(shape)` | Gaussian noise: mean=0, std=1 |
| `rng.normal(mean, std, shape)` | Gaussian with specified mean and std |
| `rng.uniform(low, high, shape)` | Uniform values in `[low, high)` |
| `rng.random(shape)` | Uniform values in `[0, 1)` — useful for random masks |
| `rng.integers(low, high, size)` | Random integers in `[low, high)` |
| `rng.choice(array, size)` | Sample `size` elements from `array` |

### Gaussian noise augmentation

One of your augmentation techniques adds small random perturbations to each feature:
```python
noise = rng.standard_normal(X.shape) * sigma   # sigma controls the strength
X_aug = X + noise
```
When `sigma` is small (e.g. 0.05), the noise is barely noticeable but breaks exact
similarity — enough to create a "different view" of the same data point.

In [9]:
# ── §6 Worked example ────────────────────────────────────────────────────────

# Demonstrate reproducibility: same seed → same numbers
rng_a = np.random.default_rng(42)
rng_b = np.random.default_rng(42)
rng_c = np.random.default_rng(99)  # different seed

print('rng_a (seed 42):', rng_a.standard_normal(3).round(4))
print('rng_b (seed 42):', rng_b.standard_normal(3).round(4))   # identical to rng_a
print('rng_c (seed 99):', rng_c.standard_normal(3).round(4))   # different
print()

# rng.random(shape): uniform values in [0, 1) — one per element
# If you threshold at 0.3, you get True with ≈30% probability per element
random_vals = rng.random(10)
print('rng.random(10):       ', random_vals.round(3))
print('rng.random(10) < 0.3: ', random_vals < 0.3)   # True with ≈30% chance each
print()

# Gaussian noise augmentation on X (120 × 6)
sigma = 0.05   # noise strength — 5% of one standard deviation (X is already normalised)
noise = rng.standard_normal(X.shape) * sigma  # shape (120, 6), values near 0
X_noisy = X + noise

print('Original X[0]:  ', X[0].round(4))
print('X_noisy [0]:    ', X_noisy[0].round(4))   # nearly the same, with tiny noise
print('Max noise added:', np.abs(noise).max().round(4))

rng_a (seed 42): [ 0.3047 -1.04    0.7505]
rng_b (seed 42): [ 0.3047 -1.04    0.7505]
rng_c (seed 99): [ 0.0825 -0.4644  0.0505]

rng.random(10):        [0.26  0.322 0.242 0.48  0.683 0.228 0.331 0.93  0.049 0.461]
rng.random(10) < 0.3:  [ True False  True False False  True False False  True False]

Original X[0]:   [ 0.3047 -1.04    0.7505  0.9406 -1.951  -1.3022]
X_noisy [0]:     [ 0.3271 -1.0371  0.7779  0.9312 -1.9371 -1.2943]
Max noise added: 0.1824


### Exercise 6

The mini-example shows how `rng.random(shape)` can create a random binary mask.
Run it first. Then implement a complete **Gaussian noise augmentation function**:

```python
def gaussian_noise(X, rng, sigma=0.1):
    ...
```

Requirements:
- Take `X`, `rng`, and `sigma` as arguments
- Return a **copy** of X with Gaussian noise added
- The noise array must have the same shape as X
- Verify: the function with `sigma=0` should return something numerically equal to X

In [ ]:
# ── §6 Mini-example: rng.random as a mask generator ─────────────────────────

# rng.random(n) gives n values uniformly between 0 and 1.
# Comparing to a threshold gives a boolean mask.

n_features = 6
mask_prob = 0.3   # each feature is masked with 30% probability

mask = rng.random(n_features) < mask_prob
print('Random mask (prob=0.3):', mask)
# On average, about 2 of the 6 values will be True (30% of 6 = 1.8)
# But the exact count varies each time — that is the randomness.

print('Number of True values: ', mask.sum(), '(expected ≈ 1.8)')

In [ ]:
# ── §6 Exercise 6 — template ─────────────────────────────────────────────────

def gaussian_noise(X, rng, sigma=0.1):
    X_aug = X.copy()          # start from a copy!
    noise = ...               # ← rng.standard_normal(X.shape) * sigma
    X_aug = X_aug + noise
    return X_aug


# Test 1: sigma=0.1, should add small noise
X_noisy = gaussian_noise(X, rng, sigma=0.1)
print('Original X[0]:  ', X[0].round(4))
print('X_noisy  [0]:   ', X_noisy[0].round(4))
print('Max noise:      ', np.abs(X_noisy - X).max().round(4))
print()

# Test 2: sigma=0, should return values equal to X
X_zero_noise = gaussian_noise(X, rng, sigma=0.0)
print('sigma=0 → max difference from X:', np.abs(X_zero_noise - X).max())
# Should be 0.0 exactly

# Test 3: original X must not be modified
X_before = X[0, 0]
_ = gaussian_noise(X, rng, sigma=1.0)
print('X[0,0] unchanged after calling function:', X[0, 0] == X_before)

In [ ]:
# ── §6 SOLUTION ──────────────────────────────────────────────────────────────

def gaussian_noise(X, rng, sigma=0.1):
    X_aug = X.copy()
    noise = rng.standard_normal(X.shape) * sigma
    X_aug = X_aug + noise
    return X_aug


X_noisy = gaussian_noise(X, rng, sigma=0.1)
print('Original X[0]:  ', X[0].round(4))
print('X_noisy  [0]:   ', X_noisy[0].round(4))
print('Max noise:      ', np.abs(X_noisy - X).max().round(4))
print()

X_zero_noise = gaussian_noise(X, rng, sigma=0.0)
print('sigma=0 → max difference from X:', np.abs(X_zero_noise - X).max())

X_before = X[0, 0]
_ = gaussian_noise(X, rng, sigma=1.0)
print('X[0,0] unchanged after calling function:', X[0, 0] == X_before)

---
## ★ Capstone: Feature Masking Augmentation (SCARF-style)

### What SCARF does

**SCARF** (Self-supervised Contrastive learning using Random Feature Corruption)
is the central augmentation technique in your thesis (Bahri et al., 2022).

For each data point, SCARF randomly corrupts some features by replacing them
with random values drawn from that feature's distribution across the dataset
(its "marginal distribution"). The model must learn that the original and corrupted
versions of the same data point are similar — which forces it to capture the
true cluster structure rather than memorising feature values.

**Simplified version (what you implement here):** instead of sampling from
marginal distributions, zero out the selected features. This is conceptually identical
and sufficient to understand the mechanics.

### Your task

Implement `feature_masking(X, rng, mask_prob=0.3)` that:

1. Makes a copy of `X` (never modify the original!)
2. For **each sample** (row), draws a random boolean mask where each feature
   is `True` with probability `mask_prob`
3. Sets the masked features to `0.0`
4. Returns the augmented array

Then verify:
- Output has the same shape as input
- `mask_prob=0.0` → no change (all zeros in noise)
- `mask_prob=1.0` → all features zeroed out
- Original `X` is unchanged

**Hint:** you need §3 (boolean indexing to zero features) and §6 (`rng.random` for the mask).

In [ ]:
# ── Capstone — YOUR CODE ─────────────────────────────────────────────────────

def feature_masking(X, rng, mask_prob=0.3):
    """
    Simplified SCARF feature masking augmentation.
    Zeroes out each feature independently with probability mask_prob.

    Parameters
    ----------
    X         : numpy array of shape (n_samples, n_features)
    rng       : numpy random generator (np.random.default_rng(seed))
    mask_prob : float in [0, 1] — probability of masking each feature

    Returns
    -------
    X_aug : numpy array of same shape as X
    """
    X_aug = X.copy()                     # Step 1: independent copy
    n_samples, n_features = X.shape

    for i in range(n_samples):           # Step 2: one mask per sample
        feat_mask = ...                  # ← rng.random(n_features) < mask_prob
        X_aug[i, feat_mask] = 0.0       # Step 3: zero out masked features

    return X_aug


# ── Tests ────────────────────────────────────────────────────────────────────

X_aug = feature_masking(X, rng, mask_prob=0.3)

# 1. Shape must match
print('Shape check (should be True):', X_aug.shape == X.shape)

# 2. Original unchanged
X_ref = X[0, 0]
_ = feature_masking(X, rng, mask_prob=0.9)
print('X unchanged (should be True):', X[0, 0] == X_ref)

# 3. mask_prob=0.0 → nothing zeroed
X_aug_zero = feature_masking(X, rng, mask_prob=0.0)
print('mask_prob=0 → no change (should be True):', np.allclose(X_aug_zero, X))

# 4. mask_prob=1.0 → everything zeroed
X_aug_full = feature_masking(X, rng, mask_prob=1.0)
print('mask_prob=1 → all zeros (should be True):', np.allclose(X_aug_full, 0.0))

# 5. Fraction of zeroed features should be close to mask_prob
X_aug_30 = feature_masking(X, rng, mask_prob=0.3)
frac_zeroed = (X_aug_30 == 0.0).mean()
print(f'Fraction zeroed with mask_prob=0.3: {frac_zeroed:.3f} (expected ≈ 0.300)')

In [ ]:
# ── Capstone — SOLUTION ──────────────────────────────────────────────────────

def feature_masking(X, rng, mask_prob=0.3):
    X_aug = X.copy()
    n_samples, n_features = X.shape

    for i in range(n_samples):
        feat_mask = rng.random(n_features) < mask_prob   # True with prob mask_prob
        X_aug[i, feat_mask] = 0.0

    return X_aug


# ── Tests ────────────────────────────────────────────────────────────────────
X_aug = feature_masking(X, rng, mask_prob=0.3)
print('Shape check (should be True):', X_aug.shape == X.shape)

X_ref = X[0, 0]
_ = feature_masking(X, rng, mask_prob=0.9)
print('X unchanged (should be True):', X[0, 0] == X_ref)

X_aug_zero = feature_masking(X, rng, mask_prob=0.0)
print('mask_prob=0 → no change (should be True):', np.allclose(X_aug_zero, X))

X_aug_full = feature_masking(X, rng, mask_prob=1.0)
print('mask_prob=1 → all zeros (should be True):', np.allclose(X_aug_full, 0.0))

X_aug_30 = feature_masking(X, rng, mask_prob=0.3)
frac_zeroed = (X_aug_30 == 0.0).mean()
print(f'Fraction zeroed with mask_prob=0.3: {frac_zeroed:.3f} (expected ≈ 0.300)')
print()

# ── Bonus: vectorised version (no Python loop — faster for large datasets) ───
def feature_masking_fast(X, rng, mask_prob=0.3):
    X_aug = X.copy()
    # rng.random(X.shape) creates a (n_samples, n_features) array in one call
    # Each element is True with probability mask_prob → a different mask per sample
    mask = rng.random(X.shape) < mask_prob
    X_aug[mask] = 0.0    # boolean indexing on a 2D array — same as the loop, but faster
    return X_aug

X_aug_fast = feature_masking_fast(X, rng, mask_prob=0.3)
frac_fast = (X_aug_fast == 0.0).mean()
print(f'Vectorised version — fraction zeroed: {frac_fast:.3f} (expected ≈ 0.300)')
print('Both versions produce valid augmentations.')